# Hợp nhất và tăng cường dữ liệu

Notebook đọc các CSV trong `data/` theo chunk, tạo dữ liệu phe sáng bằng phép đối xứng bàn cờ và ghi các tập dẫn xuất mà không giữ toàn bộ dữ liệu trong RAM.

## 1. Cấu hình

Pipeline nặng chỉ chạy khi đặt `RUN_DATA_PIPELINE=1`. Mặc định notebook chỉ kiểm tra nguồn và minh họa phép biến đổi.

In [1]:
from pathlib import Path
import os

import pandas as pd

# Thư mục gốc chứa toàn bộ CSV dữ liệu ván đấu; đường dẫn tương đối tính từ
# nơi notebook được chạy (repo root), khớp với chỗ scripts/manager.py ghi log.
DATA_DIR = Path('data')
# Các file dark_*.csv là dữ liệu ván gốc (do máy tự đấu sinh ra, xem
# scripts/log.py), có thể đã bị split.ipynb chia nhỏ thành nhiều phần; sort
# theo tên để đọc đúng thứ tự số thứ tự (dark_1, dark_2, ...).
SOURCE_FILES = sorted(DATA_DIR.glob('dark_*.csv'))
# Ba file đích: dark.csv gộp nguyên trạng các dark_*.csv; light.csv là bản đối
# xứng 180 độ + đổi phe (xem transform_chunk); animal_chess.csv là bản gộp cả
# hai phe dùng để huấn luyện model (bỏ cột id vì không cần truy vết ván).
DARK_FILE = DATA_DIR / 'dark.csv'
LIGHT_FILE = DATA_DIR / 'light.csv'
COMBINED_FILE = DATA_DIR / 'animal_chess.csv'
# Đọc/ghi CSV theo chunk để không nạp toàn bộ dữ liệu (có thể hàng GB) vào RAM
# một lần; kích thước chunk chỉnh được qua biến môi trường CSV_CHUNK_SIZE.
CHUNK_SIZE = int(os.getenv('CSV_CHUNK_SIZE', '250000'))
# Cổng an toàn: pipeline ghi file thật (tốn thời gian, ghi đè dữ liệu) chỉ
# chạy khi đặt RUN_DATA_PIPELINE=1; mặc định notebook chỉ preview (xem cell
# cuối) để tránh vô tình ghi đè dữ liệu khi chỉ mở notebook xem logic.
RUN_PIPELINE = os.getenv('RUN_DATA_PIPELINE') == '1'

# Kiểm tra sớm: không có file nguồn thì dừng ngay thay vì chạy tới lúc ghi
# file mới phát hiện ra không có gì để gộp.
if not SOURCE_FILES:
    raise FileNotFoundError('Không tìm thấy data/dark_*.csv')
# Xem nhanh danh sách file nguồn và dung lượng (GB) để ước lượng thời gian
# chạy pipeline trước khi bật RUN_DATA_PIPELINE=1.
[(path.name, round(path.stat().st_size / 1024**3, 2)) for path in SOURCE_FILES]

[('dark_1.csv', 1.75), ('dark_2.csv', 1.75), ('dark_3.csv', 0.36)]

## 2. Hàm biến đổi

Các hàm dưới đây bảo toàn schema và đảo đồng thời vị trí, phe, ký hiệu quân, điểm số.

In [2]:
# Bảng dịch ký tự dùng cho phép quay 180 độ toạ độ nước đi: cột A-G đảo thành
# G-A, hàng 1-9 đảo thành 9-1. str.translate áp dụng cho từng ký tự độc lập
# nên cùng một bảng dịch được dùng cho cả 4 ký tự của move (ô đi + ô đến).
MOVE_TRANSLATION = str.maketrans('ABCDEFG123456789', 'GFEDCBA987654321')
# Các cột số cần đổi dấu khi đổi phe: side/winner biểu diễn phe bằng dấu
# (dương/âm), trap/den là toạ độ ô đặc biệt được mã hoá có dấu, score là điểm
# đánh giá thế cờ theo góc nhìn phe tối dương/phe sáng âm (quy ước dấu này
# nằm ở Bot.evaluate_position/Log.insert_chess_record, xem scripts/log.py -
# scripts/encoding.py chỉ mã hoá tensor bàn cờ, không định nghĩa dấu điểm)
# nên đổi phe bắt buộc phải đổi dấu đồng thời cả 5 cột.
INVERT_COLUMNS = ['side', 'trap', 'den', 'score', 'winner']

def transform_board(board: str) -> str:
    # Đảo thứ tự 63 ô và đổi hoa/thường để hoán đổi hai phe.
    if len(board) != 63:
        raise ValueError(f'Bàn cờ phải có 63 ký tự, nhận {len(board)}')
    # Đảo chuỗi 63 ký tự tương đương xoay bàn cờ 180 độ (ô đầu đổi chỗ ô cuối
    # theo đúng thứ tự duyệt cố định của dataset); swapcase() đổi hoa <->
    # thường tương đương đổi quân phe tối <-> phe sáng vì ký hiệu quân dùng
    # hoa/thường để phân biệt phe (xem scripts/encoding.py PIECE_SYMBOLS).
    return board[::-1].swapcase()

def transform_move(move: str) -> str:
    # A1G9 <-> G9A1 theo phép quay 180 độ.
    if len(move) != 4:
        raise ValueError(f'Nước đi không hợp lệ: {move!r}')
    # move có dạng "<cột><hàng><cột><hàng>" (ô đi nối ô đến); translate ánh xạ
    # từng ký tự nên tự xoay cả hai toạ độ cùng lúc mà vẫn giữ đúng thứ tự
    # đi->đến, vì phép quay 180 độ là đối xứng tâm nên không cần đảo cặp
    # ô đi/đến cho nhau.
    return move.translate(MOVE_TRANSLATION)

def transform_chunk(dark: pd.DataFrame) -> pd.DataFrame:
    # Tạo dữ liệu "phe sáng" từ dữ liệu "phe tối" bằng phép đối xứng tâm bàn
    # cờ: xoay bàn 180 độ + đổi phe quân + đổi dấu các cột số liên quan. Đây
    # chính là phép tăng cường dữ liệu (data augmentation) nhân đôi số ván mà
    # không cần sinh thêm ván đấu thật, vì luật chơi đối xứng hoàn toàn giữa
    # hai phe.
    light = dark.copy()
    light['board'] = dark['board'].map(transform_board)
    light['move'] = dark['move'].map(transform_move)
    # Ký hiệu quân (r/c/d/.../R/C/D/...) chỉ cần đổi hoa/thường, không cần
    # đảo thứ tự như board vì đây là một ký tự đơn, không phải chuỗi toạ độ.
    light['piece'] = dark['piece'].str.swapcase()
    # Đổi dấu đồng loạt các cột góc nhìn theo phe (xem INVERT_COLUMNS ở trên).
    light[INVERT_COLUMNS] = -dark[INVERT_COLUMNS]
    return light

In [3]:
# Lấy thử 5 dòng đầu của file nguồn đầu tiên để kiểm tra trực quan phép biến
# đổi transform_chunk có đúng như kỳ vọng trước khi chạy toàn bộ pipeline.
sample = pd.read_csv(SOURCE_FILES[0], nrows=5)
sample_light = transform_chunk(sample)
# So sánh cạnh nhau bản gốc (dark) và bản biến đổi (light): board chỉ in 14
# ký tự đầu (kèm dấu … cho gọn) vì chuỗi 63 ký tự quá dài để đọc trực quan;
# move và score in đầy đủ vì ngắn và là chỗ dễ phát hiện lỗi dấu/toạ độ nhất.
pd.DataFrame({
    'board_dark': sample['board'].str[:14] + '…',
    'board_light': sample_light['board'].str[:14] + '…',
    'move_dark': sample['move'],
    'move_light': sample_light['move'],
    'score_dark': sample['score'],
    'score_light': sample_light['score'],
})

,board_dark,board_light,move_dark,move_light,score_dark,score_light
0,l-r---E-T-d---…,l-----E-T-dr--…,G7F7,A3B3,0,0
1,l--r--E-T-d---…,l-----E-T-dr--…,A3A4,G7G6,0,0
2,l--r--E-T-d---…,l-r---E-T-d---…,F7G7,B3A3,0,0
3,l---r-E-T-d---…,l-r---E-T-d---…,A4A5,G6G5,0,0
4,l---r--ET-d---…,l-r---E-T-d---…,A7A8,G3G2,0,0


## 3. Ghi dữ liệu theo chunk

File đích được xóa trước một lần; mỗi chunk sau đó được nối tiếp với header chính xác.

In [4]:
def merge_and_augment(source_files: list[Path]) -> dict[str, int]:
    outputs = (DARK_FILE, LIGHT_FILE, COMBINED_FILE)
    # Xoá file đích cũ (nếu có) đúng một lần trước khi ghi, vì vòng lặp bên
    # dưới ghi nối tiếp (mode='a') theo từng chunk; nếu không xoá trước, dữ
    # liệu cũ và mới sẽ bị nối lẫn vào nhau ở mỗi lần chạy lại notebook.
    for output in outputs:
        output.unlink(missing_ok=True)

    counts = {'dark': 0, 'light': 0, 'combined': 0}
    for source in source_files:
        # Đọc theo chunk (CHUNK_SIZE dòng/lần) để giới hạn RAM sử dụng dù file
        # nguồn có thể lên tới hàng GB; mỗi chunk được biến đổi và ghi ra đĩa
        # ngay rồi giải phóng khỏi bộ nhớ trước khi đọc chunk kế tiếp.
        for dark in pd.read_csv(source, chunksize=CHUNK_SIZE):
            light = transform_chunk(dark)
            # header=not <file>.exists(): chỉ ghi dòng tiêu đề CSV ở lần ghi
            # đầu tiên (khi file đích chưa tồn tại); các lần ghi nối tiếp sau
            # đó bỏ header để không lặp lại tiêu đề giữa file.
            dark.to_csv(DARK_FILE, mode='a', header=not DARK_FILE.exists(), index=False)
            light.to_csv(LIGHT_FILE, mode='a', header=not LIGHT_FILE.exists(), index=False)

            # Tập huấn luyện không cần UUID của từng ván.
            # Gộp cả hai phe (dark + light) thành một tập duy nhất để huấn
            # luyện; bỏ cột 'id' (UUID định danh ván đấu, xem scripts/log.py)
            # vì train.ipynb chỉ cần board/score, không cần truy vết dòng nào
            # sinh ra từ ván nào.
            combined = pd.concat(
                [dark.drop(columns='id'), light.drop(columns='id')],
                ignore_index=True,
            )
            combined.to_csv(
                COMBINED_FILE, mode='a', header=not COMBINED_FILE.exists(), index=False
            )
            counts['dark'] += len(dark)
            counts['light'] += len(light)
            counts['combined'] += len(combined)
    return counts

# Cổng an toàn RUN_PIPELINE (xem cell cấu hình): mặc định chỉ in thông báo
# preview, không đụng tới file trên đĩa; đặt RUN_DATA_PIPELINE=1 mới thực sự
# chạy merge_and_augment và ghi ba file dark.csv/light.csv/animal_chess.csv.
if RUN_PIPELINE:
    print(merge_and_augment(SOURCE_FILES))
else:
    print('Chế độ preview: đặt RUN_DATA_PIPELINE=1 để tạo ba file dẫn xuất.')

Chế độ preview: đặt RUN_DATA_PIPELINE=1 để tạo ba file dẫn xuất.


## Tóm tắt

- Đọc theo chunk để giới hạn RAM.
- Kiểm tra độ dài bàn cờ/nước đi trước khi biến đổi.
- Dữ liệu gốc trong `data/dark_*.csv` không bị chỉnh sửa.
- Ba file đầu ra là `dark.csv`, `light.csv` và `animal_chess.csv`.